# Feature engineering, split and preprocessor (Person 2)

CAP 3764, Fall 2026, Team 6. Michael Font (Person 2).

This notebook runs and explains `src/features.py`, which turns the cleaned product table from `src/clean.py` into model-ready data for Submission 2. It covers:

1. The minimum-ratings rule for the target
2. The 12 model features
3. The single 80/20 train/test split, stratified by category
4. The preprocessor that every model pipeline starts with

Target: `average_rating` (1-5 stars), a regression problem. Run with the **Python (amazon-reviews)** kernel, after `notebooks/02_cleaning_p2.ipynb` has saved `data/processed/products_clean.parquet`. Restart the kernel and run all cells.

In [1]:
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import pandas as pd
from src import features as F

pd.set_option("display.float_format", "{:.3f}".format)
clean = pd.read_parquet(F.CLEAN_PATH, columns=F.CLEAN_COLUMNS)
print("Cleaned rows:", f"{len(clean):,}")

Cleaned rows: 1,644,621


## 1. Minimum number of ratings

`average_rating` is the mean of a product's ratings. With only one or two ratings it mostly reflects a single buyer, not the product. The table below shows how much the target varies by rating volume across all cleaned products.

In [2]:
bands = pd.cut(clean["rating_number"], [0, 1, 4, 9, 49, float("inf")],
               labels=["1", "2-4", "5-9", "10-49", "50+"])
volume = clean.groupby(bands, observed=True)["average_rating"].agg(["count", "mean", "std"])
volume["share"] = volume["count"] / len(clean)
volume

,count,mean,std,share
rating_number,,,,
1,180016,3.997,1.481,0.109
2-4,289198,4.150,0.964,0.176
5-9,237770,4.157,0.690,0.145
10-49,506564,4.211,0.504,0.308
50+,431073,4.411,0.336,0.262


Products with a single rating have a standard deviation of about 1.48 stars, more than twice that of products with 5-9 ratings (about 0.69). Their averages are also lower, because one unhappy buyer can set the whole score.

**Decision:** keep products with at least `MIN_RATINGS = 5` ratings. This removes the noisiest targets while keeping most of the data. The cost is that the model does not describe brand-new or very niche products (see Limitations).

In [3]:
for t in [1, 2, 3, 5, 10]:
    kept = (clean["rating_number"] >= t).sum()
    print(f">= {t:>2} ratings: {kept:>9,} products ({kept / len(clean):.1%})")

>=  1 ratings: 1,644,621 products (100.0%)
>=  2 ratings: 1,464,605 products (89.1%)
>=  3 ratings: 1,342,326 products (81.6%)
>=  5 ratings: 1,175,407 products (71.5%)
>= 10 ratings:   937,637 products (57.0%)


## 2. Build the model table

`build_features()` applies the threshold, adds three listing-detail features and returns one row per product: the ID, the 12 features and the target. It does **not** impute anything. Missing prices stay missing here and are handled inside the preprocessor, so the fill value is learned from training rows only.

In [4]:
model_df = F.build_features(clean)
print("Model rows:", f"{len(model_df):,}", "| columns:", model_df.shape[1])
model_df.head()

Model rows: 1,175,407 | columns: 14


,parent_asin,log_price,log_rating_number,n_features,n_description,title_len,n_categories,n_details,has_price,price_is_from,category,main_category,store,average_rating
0,B01CUPMQZE,NaN,2.398,0,0,51,0,2,0,0,All_Beauty,All Beauty,Howard Products,4.800
1,B000B658RI,NaN,3.296,0,0,48,0,1,0,0,All_Beauty,All Beauty,Levine Health Products,4.400
2,B088FKY3VD,NaN,4.635,0,0,158,0,7,0,0,All_Beauty,All Beauty,Cherioll,3.100
3,B07NGFDN6G,NaN,2.079,5,1,70,0,1,0,0,All_Beauty,All Beauty,Precision,4.300
4,B07G9GWFSM,2.078,3.584,5,6,89,0,8,1,0,All_Beauty,All Beauty,Lurrose,3.700


| Feature | Type | Source | Why it may relate to ratings |
|---|---|---|---|
| `log_price` | numeric | cleaning (`log1p` of price) | Price sets buyer expectations; log reduces extreme prices |
| `log_rating_number` | numeric | cleaning (`log1p` of rating count) | Popular products tend to be rated higher (EDA) |
| `n_features` | numeric | cleaning (bullet points) | More detailed listings may set clearer expectations |
| `n_description` | numeric | cleaning (description paragraphs) | Same idea as above |
| `title_len` | numeric | new: title characters | Listing effort and detail |
| `n_categories` | numeric | new: category path depth | How specifically the product is categorized |
| `n_details` | numeric | new: keys in the details field | Completeness of product specifications |
| `has_price` | binary | cleaning | Whether a price was listed at all |
| `price_is_from` | binary | cleaning | Price shown as "from X" (several variants) |
| `category` | categorical | source tag (4 values) | Rating norms differ by category |
| `main_category` | categorical | metadata (42 values) | Finer product group |
| `store` | high-cardinality | metadata (159,052 values after the filter) | Seller reputation |

In [5]:
model_df[F.NUMERIC + F.BINARY].describe().T

,count,mean,std,min,25%,50%,75%,max
log_price,598577.000,3.158,0.984,0.010,2.524,2.995,3.624,10.820
log_rating_number,1175407.000,3.667,1.455,1.792,2.485,3.367,4.522,12.215
n_features,1175407.000,3.611,2.202,0.000,1.000,5.000,5.000,60.000
n_description,1175407.000,1.864,5.026,0.000,0.000,1.000,1.000,391.000
title_len,1175407.000,104.777,52.974,0.000,60.000,96.000,149.000,1455.000
n_categories,1175407.000,3.309,1.574,0.000,3.000,4.000,4.000,7.000
n_details,1175407.000,8.878,4.882,0.000,6.000,8.000,11.000,78.000
has_price,1175407.000,0.509,0.500,0.000,0.000,1.000,1.000,1.000
price_is_from,1175407.000,0.000,0.003,0.000,0.000,0.000,0.000,1.000


In [6]:
print("Missing values in the model table:")
print(model_df[F.FEATURES].isna().sum()[lambda s: s > 0])
print()
print("Distinct values:", {c: model_df[c].nunique() for c in F.CATEGORICAL + F.HIGH_CARDINALITY})

Missing values in the model table:
log_price    576830
dtype: int64



Distinct values: {'category': 4, 'main_category': 42, 'store': 159052}


Only `log_price` has gaps, because roughly half of the listings have no price. `has_price` records that, so the model can treat "no price listed" differently from an average price.

## 3. Train/test split

`split_data()` makes the project's only split: 80% train, 20% test, stratified by `category` so each source category keeps the same share in both sets, with seed 42. Models are tuned with cross-validation on the training set. **The test set is used once, for the final evaluation.**

In [7]:
train, test = F.split_data(model_df)
print(f"Train: {len(train):,}  Test: {len(test):,}  ({len(test) / len(model_df):.1%} test)")
shares = pd.DataFrame({
    "train": train["category"].value_counts(normalize=True),
    "test": test["category"].value_counts(normalize=True),
})
shares["difference"] = (shares["train"] - shares["test"]).abs()
shares

Train: 940,325  Test: 235,082  (20.0% test)


,train,test,difference
category,,,
Toys_and_Games,0.594,0.594,0.000
Industrial_and_Scientific,0.224,0.224,0.000
Musical_Instruments,0.117,0.117,0.000
All_Beauty,0.065,0.065,0.000


In [8]:
overlap = set(train[F.ID]) & set(test[F.ID])
print("Products in both train and test:", len(overlap))
print("Target mean  train / test:", round(train[F.TARGET].mean(), 3), "/", round(test[F.TARGET].mean(), 3))

Products in both train and test: 0
Target mean  train / test: 4.274 / 4.273


## 4. Preprocessor

`make_preprocessor()` returns an **unfitted** `ColumnTransformer`. Each model puts it first in a pipeline, so during cross-validation it is refitted on the training folds only:

- Numeric: median imputation, then standard scaling (needed for Ridge; harmless for trees)
- Binary: passed through as 0/1
- `category`, `main_category`: one-hot; main categories with fewer than 100 training rows are grouped as infrequent
- `store`: `TargetEncoder`, which replaces each store with a smoothed average rating. It cross-fits internally, so a product's own rating never feeds its own encoding. One-hot encoding about 159,000 stores would not be practical.

Below, it is fitted on the training set only to show its output. The test set is not touched.

In [9]:
pre = F.make_preprocessor().fit(train[F.FEATURES], train[F.TARGET])
names = pre.get_feature_names_out()
print("Columns after preprocessing:", len(names))
print(list(names))

Columns after preprocessing: 42
['numeric__log_price', 'numeric__log_rating_number', 'numeric__n_features', 'numeric__n_description', 'numeric__title_len', 'numeric__n_categories', 'numeric__n_details', 'binary__has_price', 'binary__price_is_from', 'categorical__category_All_Beauty', 'categorical__category_Industrial_and_Scientific', 'categorical__category_Musical_Instruments', 'categorical__category_Toys_and_Games', 'categorical__main_category_AMAZON FASHION', 'categorical__main_category_All Beauty', 'categorical__main_category_All Electronics', 'categorical__main_category_Amazon Home', 'categorical__main_category_Appliances', 'categorical__main_category_Arts, Crafts & Sewing', 'categorical__main_category_Automotive', 'categorical__main_category_Baby', 'categorical__main_category_Books', 'categorical__main_category_Camera & Photo', 'categorical__main_category_Car Electronics', 'categorical__main_category_Cell Phones & Accessories', 'categorical__main_category_Computers', 'categorical_

## 5. Training-set relationships

Spearman correlations between each numeric or binary feature and the target, on **training rows only**. These are associations, not causes, and describe single features without controlling for the others.

In [10]:
corr = train[F.NUMERIC + F.BINARY].corrwith(train[F.TARGET], method="spearman")
corr.sort_values(key=abs, ascending=False).to_frame("spearman_with_rating")

,spearman_with_rating
has_price,0.203
log_rating_number,0.175
n_categories,0.098
n_details,0.092
n_features,0.086
n_description,0.084
title_len,-0.065
log_price,-0.011
price_is_from,0.001


All single-feature associations are weak (|rho| at most about 0.2), so no one listing detail explains ratings on its own; the models combine them.

- `has_price` is the strongest: listings that show a price tend to have higher ratings. Price *level* (`log_price`) is almost unrelated. A likely explanation is that a listed price marks an active, in-stock listing, but this data cannot confirm that.
- `log_rating_number` is positive, matching the EDA: more-rated products tend to score higher. Popularity and quality can reinforce each other, so this is not evidence that more ratings cause better ones.
- More complete listings (categories, details, features, description) are slightly higher rated; longer titles are slightly lower rated.

## 6. Save the split for modeling

`build_model_data()` repeats steps 2-3 from the cleaned file and saves `train.parquet` and `test.parquet` under the ignored `data/processed/features/` folder. Models load them with `load_split()`:

```python
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from src.features import load_split, make_preprocessor

X_train, X_test, y_train, y_test = load_split()
model = make_pipeline(make_preprocessor(), Ridge())
```

In [11]:
paths = F.build_model_data()
X_train, X_test, y_train, y_test = F.load_split()
print(X_train.shape, X_test.shape)

features: 1,644,621 cleaned -> 1,175,407 with >= 5 ratings; train 940,325, test 235,082; C:\Users\Gamer\CAP_3764_2026_Fall_Team_6\data\processed\features


(940325, 12) (235082, 12)


## Feature decisions (Person 2)

| Step | Decision | Rows affected | Rows after |
|---|---|---|---|
| Input | Cleaned table from `src/clean.py` | | 1,644,621 |
| Target filter | Keep products with at least 5 ratings | 469,214 removed (28.5%) | 1,175,407 |
| New features | `title_len`, `n_categories`, `n_details` from listing fields | | 1,175,407 |
| Missing prices | Left missing; median-imputed inside the preprocessor; flagged by `has_price` | about half of rows | 1,175,407 |
| Store | Target-encoded inside the preprocessor (cross-fitted) | | 1,175,407 |
| Split | 80/20, stratified by `category`, seed 42 | | 940,325 train / 235,082 test |

Handoff: Person 3 builds models with `load_split()` and `make_preprocessor()`; Person 1 connects `build_model_data()` in the pipeline (P1-c).